# 13.9 只優化一種偏好會失去什麼？

# 第 13 章：把回答偏好變成訓練

前置：第7章SFT，以及第8–9章的「好回答」規則。可以直接走這條路，不必先學圖片。把兩篇答案像兩道菜一起試喫：喜歡哪一篇，與「兩篇是否都安全」是不同問題。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：試喫兩道菜：比起原廚師，更偏向好回答**

chosen 與 rejected 必須對同一個問題。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/dpo.svg")))

**先想一想：**安全比較全來自需要拒絕的題，會漏掉什麼？

只優化單一偏好可能犧牲其他能力。混合資料要列來源比例與有效tokens，評估也列風格、遵循、誠實、安全和原任務。

**把直覺寫成數學：**多面向metrics向量；不要讓單一勝率替代所有行為。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
recipe = {"style": 30, "format": 30, "safety": 20, "benign_completion": 20}
print(recipe, "sum", sum(recipe.values()))
metrics = {k: None for k in ("style", "format", "truth", "safe", "benign_completion", "text_task")}
print("待訓練後測量", metrics)

**如何讀結果：**None表示未量測；非商用來源需保持獨立標記。不要把來源授權混成預設全可商用。

**只改一件事：**只移除benign_completion來源，指出可能過度拒絕。
